# 03 · Repertoire size against distance from the root

Every node of the rooted human tree, the 432 reconstructed ancestors and the 433 extant
human receptors, with its number of called ligands and its patristic distance from the
root. Ancestors are called from the ASR matrix and extant receptors from the
reference-tree matrix, each by its own density fill. Writes
`results/ligands_vs_root_distance.csv` and `results/root_distance_statistics.csv` (Extended Data
Fig. 5a and 5c).

In [1]:
import os
import sys
from pathlib import Path

REPO = Path("../..").resolve()
sys.path.append(str(REPO / "Downstream_Analysis/scripts"))
sys.path.append(str(REPO / "scripts"))

from ancestral_sets import ASR_MATRIX, REF_MATRIX, PAIRS, FASTA, ASR_TREE, TABLES

INPUTS = [ASR_MATRIX, REF_MATRIX, PAIRS, FASTA, ASR_TREE]
TABLES_WRITTEN = ["ligands_vs_root_distance", "root_distance_statistics"]

for path in INPUTS:
    print("input  ", os.path.relpath(path, REPO))
print("tables ", os.path.relpath(TABLES, REPO), TABLES_WRITTEN)

input   Downstream_Analysis/predictions/aggregated/ASR_Predictions/asr_median_probability_wide.csv
input   Downstream_Analysis/predictions/aggregated/Reference_Tree_Predictions/reference_median_probability_wide.csv
input   Model_Inputs/Data_Preparation/processed/m2or_pairs_model.csv
input   Phylogenetic_Analysis/data/ReferenceTree/PF13853.9606_7955_7740_7764_75743_137246.fa
input   Ancestral_Receptor_Reconstruction/human_tree_asr_hagfish_outgroup/PF13853_9606.7764.mafft.lg.treefile.rooted.withinternalnames
tables  Downstream_Analysis/ancestral_receptors/results ['ligands_vs_root_distance', 'root_distance_statistics']


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, pearsonr, spearmanr

from ancestral_sets import decay_table, save_table

decay = decay_table()
save_table(decay, "ligands_vs_root_distance")
print(decay.groupby("kind").agg(n=("name", "size"),
                                median_ligands=("n_ligands", "median"),
                                median_distance=("distance", "median")).round(2).to_string())

            n  median_ligands  median_distance
kind                                          
ancestor  432             7.0             3.68
extant    433             4.0             4.13


## Ligand count against distance, and ancestors against extant receptors

Pearson and Spearman correlation and the least-squares line over all nodes, the same within
ancestors and within extant receptors, and a Mann–Whitney test of ancestor against extant
ligand counts. Writes `results/root_distance_statistics.csv`.

In [3]:
x, y = decay["distance"].to_numpy(), decay["n_ligands"].to_numpy()
r_p, p_p = pearsonr(x, y)
r_s, p_s = spearmanr(x, y)
slope, intercept = np.polyfit(x, y, 1)
print(f"all nodes   Pearson r = {r_p:+.3f}  p = {p_p:.3g}")
print(f"            Spearman r = {r_s:+.3f}  p = {p_s:.3g}")
print(f"            slope = {slope:+.2f} ligands per unit distance")
for kind in ("ancestor", "extant"):
    sub = decay[decay.kind == kind]
    rp, pp = pearsonr(sub.distance, sub.n_ligands)
    print(f"{kind:<11} Pearson r = {rp:+.3f}  p = {pp:.3g}   (n={len(sub)})")

ancestors = decay.loc[decay.kind == "ancestor", "n_ligands"].values
extant = decay.loc[decay.kind == "extant", "n_ligands"].values
u = mannwhitneyu(ancestors, extant)
print(f"median ligands: ancestors {np.median(ancestors):.0f}, extant {np.median(extant):.0f}, "
      f"Mann-Whitney p = {u.pvalue:.2g}")

stats = pd.DataFrame({"statistic": ["pearson_r", "pearson_p", "spearman_r", "spearman_p", "slope",
                                    "intercept", "median_ancestor", "median_extant", "mannwhitney_p"],
                      "value": [r_p, p_p, r_s, p_s, slope, intercept, np.median(ancestors),
                                np.median(extant), u.pvalue]})
save_table(stats, "root_distance_statistics")

all nodes   Pearson r = +0.058  p = 0.0884
            Spearman r = -0.121  p = 0.000377
            slope = +6.13 ligands per unit distance
ancestor    Pearson r = +0.127  p = 0.00811   (n=432)
extant      Pearson r = -0.016  p = 0.743   (n=433)
median ligands: ancestors 7, extant 4, Mann-Whitney p = 1.8e-08
